# Congressional Trading Lag Arbitrage — Exploratory Analysis

**Author:** Quantitative Research  
**Strategy:** STOCK Act Disclosure Lag Signal  

This notebook walks through every stage of the research pipeline end-to-end:
data loading → feature engineering → signal construction → OLS regression → backtest → interpretation.

All heavy lifting is delegated to `src/` modules — no logic is duplicated here.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Ensure repo root is on the path regardless of working directory
from pathlib import Path
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src import scraper, features, signal, backtest, visualize

print('Imports OK. Repo root:', repo_root)
print('Data dir:', config.DATA_DIR)
print('Outputs dir:', config.OUTPUTS_DIR)

---
## 1. Data Loading & Overview

We fetch raw disclosures from the housestockwatcher.com and senatestockwatcher.com public JSON APIs.
Responses are cached in `data/` so re-runs are instant.

In [ ]:
house_df = scraper.fetch_house_disclosures()
senate_df = scraper.fetch_senate_disclosures()
df = scraper.combine_disclosures(house_df, senate_df)

print(f'Combined shape: {df.shape}')
print(f'Date range: {df["disclosure_date"].min().date()} → {df["disclosure_date"].max().date()}')
print(f'\nChamber split:')
print(df['chamber'].value_counts())
print(f'\nTransaction types:')
print(df['transaction_type'].value_counts())

In [ ]:
df.head(5)

In [ ]:
print('Column dtypes:')
print(df.dtypes)
print(f'\nMissing values per column:')
print(df.isna().sum())

---
## 2. Disclosure Lag Analysis

The STOCK Act (2012) requires disclosure within **45 calendar days** of the trade.
We compute `lag_days` = `disclosure_date` − `transaction_date` and flag/drop anomalous rows.

In [ ]:
df = features.compute_disclosure_lag(df)

print('Lag summary statistics:')
print(df['lag_days'].describe().round(1))

In [ ]:
fig = visualize.plot_lag_distribution(df)
plt.show()
print(f'\nDistribution by transaction type:')
print(df.groupby('transaction_type')['lag_days'].describe().round(1))

In [ ]:
# What fraction of disclosures cluster in the final week before the deadline?
late_pct = (df['lag_days'] >= 38).mean() * 100
print(f'Disclosures filed in the last 7 days (>=38): {late_pct:.1f}%')

# Median lag by chamber
print(df.groupby('chamber')['lag_days'].median().rename('median_lag_days'))

---
## 3. Coverage Score Analysis

We scrape Finviz to count recent news articles per ticker as a proxy for **analyst coverage intensity**.
Tickers with fewer articles (low-coverage names) are where informational asymmetry is largest.

> ⚠️ **Note**: `compute_coverage_score` makes one HTTP request per unique ticker.
> For a universe of 500+ tickers this takes ~8 minutes. A cached run is instant.

In [ ]:
# Limit to a manageable subset for demonstration; remove slice for full run
top_tickers = df['ticker'].value_counts().head(50).index.tolist()

df, coverage_dict = features.compute_coverage_score(df, tickers=top_tickers)

print(f'Coverage scores computed for {len(coverage_dict)} tickers.')
print(f'\nTop 10 highest-coverage tickers:')
top_coverage = sorted(coverage_dict.items(), key=lambda x: x[1], reverse=True)[:10]
for ticker, count in top_coverage:
    print(f'  {ticker:6s}: {count} articles')

In [ ]:
fig = visualize.plot_coverage_score_distribution(df)
plt.show()

print(f'Low-coverage tickers: {df["low_coverage"].sum()} rows ({df["low_coverage"].mean()*100:.1f}%)')

---
## 4. Composite Signal Construction

We apply direction encoding, committee weighting, and finally compute the composite ranking score.

**Formula:**

$$
\text{composite} = \text{direction} \times \ln(1 + \text{lag\_days}) \times \text{committee\_weight} \times \frac{1}{1 + \text{coverage\_score}}
$$

In [ ]:
df = features.compute_committee_weight(df)
df = features.compute_direction_flag(df)
df = features.compute_composite_score(df)

print(f'Composite score stats:')
print(df['composite_score'].describe().round(4))

In [ ]:
print('Top 20 LONG signals:')
top20 = df.nlargest(20, 'composite_score')[[
    'filer_name', 'ticker', 'disclosure_date', 'lag_days',
    'coverage_score', 'committee_weight', 'direction', 'composite_score'
]]
top20

In [ ]:
print('Top 20 SHORT signals (most negative composite):')
bot20 = df.nsmallest(20, 'composite_score')[[
    'filer_name', 'ticker', 'disclosure_date', 'lag_days',
    'coverage_score', 'committee_weight', 'direction', 'composite_score'
]]
bot20

---
## 5. OLS Regression Results

We test whether the composite signal features statistically predict 10-day forward returns using OLS.

**Specification:**

$$
\text{forward\_return}_i = \alpha + \beta_1 \text{direction}_i + \beta_2 \text{lag\_days}_i + \beta_3 \text{coverage\_score}_i + \beta_4 \text{committee\_weight}_i + \varepsilon_i
$$

In [ ]:
# Fetch price data for all tickers in the dataset
all_tickers = df['ticker'].dropna().unique().tolist()

min_date = (df['disclosure_date'].min() - pd.Timedelta(days=5)).date()
max_date = (df['disclosure_date'].max() + pd.Timedelta(days=config.POST_DISCLOSURE_WINDOW * 2)).date()

print(f'Fetching prices for {len(all_tickers)} tickers ({min_date} → {max_date})...')
price_data = scraper.fetch_price_data(all_tickers, str(min_date), str(max_date))
print(f'Price data shape: {price_data.shape}')

In [ ]:
ols_model, df_enriched = signal.run_ols_regression(df, price_data)
print(ols_model.summary())

In [ ]:
# Coefficient table
import statsmodels.api as sm
coef_df = pd.DataFrame({
    'coefficient': ols_model.params,
    'std_error': ols_model.bse,
    't_stat': ols_model.tvalues,
    'p_value': ols_model.pvalues,
    'significant': ols_model.pvalues < 0.05,
})
coef_df.round(4)

In [ ]:
# Visualise composite score vs forward return
df_enriched = signal.rank_signals(df_enriched)
fig = visualize.plot_composite_score_vs_return(df_enriched)
plt.show()

---
## 6. Backtest Results

We run the event-driven long/short backtest and compute performance metrics.

In [ ]:
# Generate positions from the ranked signal
positions_df = signal.generate_weekly_positions(df_enriched)
print(f'Positions generated: {len(positions_df)} rows')
print(f'Unique weeks: {positions_df["week"].nunique()}')
print(f'\nPosition split:')
print(positions_df['position'].value_counts())

In [ ]:
results_df = backtest.run_backtest(positions_df, price_data)
print(f'Backtest weeks with PnL data: {len(results_df)}')
results_df.head(10)

In [ ]:
metrics = backtest.compute_performance_metrics(results_df)
print('\nPerformance Metrics:')
print(f'  Annualised Sharpe Ratio : {metrics["sharpe_ratio"]:+.4f}')
print(f'  Maximum Drawdown        : {metrics["max_drawdown"]:+.4f}')
print(f'  Win Rate                : {metrics["win_rate"]*100:.1f}%')
print(f'  Total Trades            : {metrics["n_trades"]}')
print(f'  Avg Holding Return      : {metrics["avg_holding_return"]:+.4f}')

In [ ]:
if len(results_df) > 0:
    fig = visualize.plot_cumulative_pnl(results_df)
    plt.show()
else:
    print('No backtest results to plot.')

---
## 7. Interpretation

### 7.1 What the signal is measuring

The composite score rewards Congressional trades that:
- Are filed **late** (close to the 45-day deadline) — suggesting the member delayed a potentially damaging or revealing disclosure.
- Come from members on **high-influence committees** (Armed Services, Intelligence) who have regulatory or procurement oversight of the traded company.
- Are in **low-coverage, obscure names** — where the information advantage of a well-connected legislator is largest relative to an under-researched market.

The hypothesis is that these three conditions jointly select for the trades most likely to be informed.

### 7.2 What to watch out for — key limitations

| Limitation | Description | Mitigation |
|---|---|---|
| **Transaction size ambiguity** | The STOCK Act only requires reporting in buckets (e.g. `$1,001 – $15,000`). A disclosure in the top bucket could represent $5M or $1M. The midpoint proxy systematically understates large trades. | Use `amount_midpoint` only for filtering, not as a signal weight. |
| **Survivorship bias in disclosure data** | Members who trade and are later investigated or scrutinised for insider trading may have had their disclosures removed or amended from the public record. | Treat the dataset as a non-random sample; performance estimates are optimistic. |
| **Look-ahead bias risk** | Coverage scores computed from Finviz at run time reflect *current* article counts, not counts at the time of the historical disclosure. For back-period rows this introduces a mild information leak. | In a production system, store time-series coverage snapshots. |
| **Small sample per cell** | OLS regression cells (e.g. Armed Services members who bought low-coverage stocks in 2022) may contain only 5–15 observations, making t-statistics unreliable. | Bootstrap or use Fama-MacBeth cross-sectional regressions for robustness. |

### 7.3 Honest assessment of the strategy

Academic evidence (Karadas et al. 2017; Eggers & Hainmueller 2013) suggests that Congressional trades **on average** outperform the market, but the effect is concentrated in specific committees, time periods, and stock types. The STOCK Act has materially reduced the advantage by compressing the disclosure window.

This repository's backtest should be treated as a **research prototype**, not a live trading signal:
- Bid/ask spreads on small-cap, low-coverage names can easily wipe out reported alpha.
- Position sizes implied by the disclosure buckets are small — realistic after-cost alpha may be near zero.
- The signal is best used as a **screening tool** to flag names for further fundamental research, not as a mechanical rule.

### 7.4 Next steps

1. **Time-series coverage snapshots** — replace Finviz scraping with historical IBES analyst count data.
2. **Transaction cost modelling** — apply half-spread and market impact estimates per size bucket.
3. **Factor attribution** — regress strategy returns on Fama-French 5-factor model to identify true alpha vs. beta exposure.
4. **NLP on committee hearings** — enrich the signal with topic modelling of Congressional hearings to identify which sectors a committee is actively engaged with at the time of each trade.

---
*This notebook is for academic and educational purposes only. See `README.md` for full disclaimer.*